# 2026-03-06 - Exported CSVs for Bapun Sessions:
## Turbo: `'/nfs/turbo/umms-kdiba/Bapun/2026-03-06_BapunSessionsExport.csv'`
## locker_DataDen: `'/home/halechr/cloud/locker_dataDen/Data/Bapun/2026-03-06_BapunSessionsExport_DataDen.csv'`

bump


In [2]:
# Symlinkers
import sys
import os
import shutil
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from typing import Dict, Optional, Tuple
from attrs import define, field, Factory
from pyphocorehelpers.Filesystem.symbolic_link_helpers import SymlinkManager
from pyphocorehelpers.Filesystem.symbolic_link_helpers import make_specific_items_local



In [3]:
# data_folder_path = Path(r'/home/halechr/cloud/turbo/Data').resolve()
data_folder_path = Path('/home/halechr/cloud/locker_dataDen/Data/').resolve()

assert data_folder_path.exists() and data_folder_path.is_dir()

bapun_path = data_folder_path.joinpath('Bapun').resolve()
bapun_path
assert bapun_path.exists() and bapun_path.is_dir()
# bapun_path.glob(



In [6]:
import pandas as pd
from pathlib import Path

def walk_dirs_filtered(base_path, max_depth, exclude_list, _current_depth=1):
    """Yields directories up to a depth, ignoring specified folders and their subpaths."""
    for item in base_path.iterdir():
        if item.is_dir():
            # If the folder name is in our exclude list, skip it entirely
            if item.name in exclude_list:
                continue
            
            yield item
            
            # Only recurse deeper if we haven't hit the max depth
            if _current_depth < max_depth:
                yield from walk_dirs_filtered(item, max_depth, exclude_list, _current_depth + 1)

base_dir = bapun_path # Path("C:/your/target/directory")
exclude_list = ['misc'] # , '', 'CRCNSData'

# Generate the data
data = [
    {'name': d.name, 'path': str(d.resolve())} 
    for d in walk_dirs_filtered(base_dir, max_depth=2, exclude_list=exclude_list)
]

df = pd.DataFrame(data, columns=['name', 'path'])
print(df)

                       name                                               path
0        331D_F_drive_check  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
1                      RatJ  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
2                      POST  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
3                  position  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
4          ReclusteredDatas  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
..                      ...                                                ...
60             DropboxFiles  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
61         3DmodelPrototype  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
62  GrosmarkReclusteredData  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
63                 3DModels  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
64            MultiMazeData  /home/halechr/cloud/locker_dataDen/Data/Bapun/...

[65 rows x 2 columns]


In [7]:
# Create a function to extract the hierarchy relative to the base directory
def get_levels(path_str):
    rel_parts = Path(path_str).relative_to(base_dir.resolve()).parts
    top_level = rel_parts[0] if len(rel_parts) > 0 else None
    sub_level = rel_parts[1] if len(rel_parts) > 1 else ""
    return pd.Series([top_level, sub_level])

# Apply the function to create hierarchy columns
df[['TopLevel', 'Level2']] = df['path'].apply(get_levels)

# Set the multi-index and sort it
df_hierarchical: pd.DataFrame = df.set_index(['TopLevel', 'Level2']).sort_index()

# Drop the original 'name' column if it feels redundant now
df_hierarchical = df_hierarchical.drop(columns=['name'])

# print(df_hierarchical)
df_hierarchical

path
TopLevel                  Level2                                                                    
331D_F_drive_check                                 /home/halechr/cloud/locker_dataDen/Data/Bapun/...
                          POST                     /home/halechr/cloud/locker_dataDen/Data/Bapun/...
                          RatJ                     /home/halechr/cloud/locker_dataDen/Data/Bapun/...
                          position                 /home/halechr/cloud/locker_dataDen/Data/Bapun/...
Hiro_10hrPost_reclustered                          /home/halechr/cloud/locker_dataDen/Data/Bapun/...
...                                                                                              ...
mbox_data                 ChannelMapRepo           /home/halechr/cloud/locker_dataDen/Data/Bapun/...
                          DropboxFiles             /home/halechr/cloud/locker_dataDen/Data/Bapun/...
                          ExperimentSetup          /home/halechr/cloud/locker_dataDen/Data/Bapun/...
                          GrosmarkReclusteredData  /home/halechr/cloud/locker_dataDen/Data/Bapun/...
                          MultiMazeData            /home/halechr/cloud/locker_dataDen/Data/Bapun/...

[65 rows x 1 columns]

In [8]:
# export_csv_path: Path = base_dir.joinpath(f'2026-03-06_BapunSessionsExport.csv')
export_csv_path: Path = base_dir.joinpath(f'2026-03-06_BapunSessionsExport_DataDen.csv')

df.to_csv(export_csv_path) # '/nfs/turbo/umms-kdiba/Bapun/2026-03-06_BapunSessionsExport.csv'
export_csv_path

PosixPath('/home/halechr/cloud/locker_dataDen/Data/Bapun/2026-03-06_BapunSessionsExport_DataDen.csv')